In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
df = pd.read_csv("../data/Telco-Customer-Churn.csv")


In [4]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [5]:
df = df.drop(columns=["customerID"])
X = df.drop(columns=["Churn"])
y = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

In [6]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("Target distribution:")
print(y.value_counts())


Features shape: (7043, 19)
Target shape: (7043,)
Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [9]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns


In [10]:
print("Numerical columns:")
print(list(num_cols))

print("\nCategorical columns:")
print(list(cat_cols))


Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [12]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [13]:
from sklearn.linear_model import LogisticRegression

In [14]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [15]:
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

In [17]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [18]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("Model Evaluation Results")
print("------------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("ROC-AUC:", round(roc_auc, 4))

Model Evaluation Results
------------------------
Accuracy: 0.8055
Precision: 0.6572
Recall: 0.5588
ROC-AUC: 0.8419


In [19]:
import joblib

In [20]:
# Save the complete trained pipeline

joblib.dump(model, "../models/churn_pipeline.pkl")

print("Model saved successfully!")

Model saved successfully!


In [23]:
# Day 2

In [21]:
print("Checking Day 1 components...")

print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

print(preprocessor)

Checking Day 1 components...
(5634, 19)
(1409, 19)
(5634,)
(1409,)
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'Tec

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

In [24]:

# DAY 2 - Model Comparison

RANDOM_STATE = 42

# Define three candidate models
day2_candidates = {

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

print("Candidate models:", list(day2_candidates))


Candidate models: ['Decision Tree', 'Random Forest', 'SVM']


In [25]:
# DAY 2 - Cross Validation

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [26]:
day2_cv_results = {}

for model_name, estimator in day2_candidates.items():

    candidate_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])

    scores = cross_val_score(
        candidate_pipeline,
        X_train,
        y_train,
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1
    )

    day2_cv_results[model_name] = scores

    print(f"\n{model_name}")
    print("Fold Recall:", scores)
    print("Mean Recall:", scores.mean())


Decision Tree
Fold Recall: [0.50803213 0.47389558 0.49098196]
Mean Recall: 0.4909698915904097

Random Forest
Fold Recall: [0.62248996 0.65461847 0.61923848]
Mean Recall: 0.6321156368962825

SVM
Fold Recall: [0.77309237 0.76907631 0.79559118]
Mean Recall: 0.7792532856878416


In [27]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

day2_trained_models = {}
day2_test_results = {}

for model_name, estimator in day2_candidates.items():

    candidate_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])

    # Train
    candidate_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = candidate_pipeline.predict(X_test)
    y_prob = candidate_pipeline.predict_proba(X_test)[:, 1]

    # Evaluate
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

    day2_trained_models[model_name] = candidate_pipeline
    day2_test_results[model_name] = metrics

    print(f"\n{model_name}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")



Decision Tree
Accuracy: 0.7317
Precision: 0.4946
Recall: 0.4866
ROC-AUC: 0.6537

Random Forest
Accuracy: 0.7672
Precision: 0.5542
Recall: 0.6283
ROC-AUC: 0.8224


c:\python312\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



SVM
Accuracy: 0.7438
Precision: 0.5115
Recall: 0.7727
ROC-AUC: 0.8211


In [29]:

import joblib
from pathlib import Path

# Load Day 1 model
baseline_model = joblib.load(
    Path("../models/churn_pipeline.pkl")
)

# Predict using Day 1 model
baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

# Calculate baseline metrics
day1_results = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "ROC-AUC": roc_auc_score(y_test, baseline_prob)
}

print("Day 1 Logistic Regression")
for metric, value in day1_results.items():
    print(f"{metric}: {value:.4f}")


Day 1 Logistic Regression
Accuracy: 0.8055
Precision: 0.6572
Recall: 0.5588
ROC-AUC: 0.8419


In [30]:
comparison = {
    "Logistic Regression": day1_results,
    **day2_test_results
}

comparison_df = pd.DataFrame(comparison).T

print(comparison_df.round(4))

                     Accuracy  Precision  Recall  ROC-AUC
Logistic Regression    0.8055     0.6572  0.5588   0.8419
Decision Tree          0.7317     0.4946  0.4866   0.6537
Random Forest          0.7672     0.5542  0.6283   0.8224
SVM                    0.7438     0.5115  0.7727   0.8211


In [31]:
# Select champion based on mean CV Recall

champion_name = max(
    day2_cv_results,
    key=lambda name: day2_cv_results[name].mean()
)

print("Champion Model:", champion_name)
print(
    "Mean CV Recall:",
    round(day2_cv_results[champion_name].mean(), 4)
)

Champion Model: SVM
Mean CV Recall: 0.7793


In [32]:
import joblib
from pathlib import Path

# Get the already trained champion pipeline
champion_model = day2_trained_models[champion_name]

# Save alongside Day 1 model
model_path = Path("../models/churn_day2_champion.joblib")

model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(champion_model, model_path)

print("Champion saved successfully!")
print("Location:", model_path)

Champion saved successfully!
Location: ..\models\churn_day2_champion.joblib
